# Segmentation Evaluation — PhiSat2 & segTHRawS

Evaluates Majority Class, Random Forest, XGBoost and U-Net at pixel level:
- **Metrics table** at threshold 0.5 and 0.85: accuracy, precision, recall, F1, mIoU
- **mIoU vs threshold** combined plot
- **Precision & Recall vs threshold** per model
- **Full classification reports** at threshold 0.5 and 0.85

Set `DATASET` to either `'PhiSat2'` or `'segTHRawS'` to switch between datasets.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    jaccard_score, precision_score, recall_score,
    f1_score, accuracy_score, classification_report
)

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATASET    = 'segTHRawS'   # Switch to 'segTHRawS' for the other dataset
BASE_DIR   = '/home/laura/Scriptie/bewerkte_data'
THRESHOLDS = np.linspace(0.05, 0.95, 19)
# ────────────────────────────────────────────────────────────────────────────

data_dir = os.path.join(BASE_DIR, DATASET)
print(f"Dataset: {DATASET}")
print(f"Data directory: {data_dir}")

In [ ]:
# Load data
# Baselines and traditional models (from seg_models_phiSat_segTHRawS.ipynb)
y_test          = np.load(os.path.join(data_dir, f'y_test_{DATASET}.npy')).flatten()
y_prob_majority = np.load(os.path.join(data_dir, f'y_prob_majority_{DATASET}.npy')).flatten()
y_prob_rf       = np.load(os.path.join(data_dir, f'y_prob_rf_{DATASET}.npy')).flatten()
y_prob_xgb      = np.load(os.path.join(data_dir, f'y_prob_xgb_{DATASET}.npy')).flatten()

# U-Net has its own validation set (fixed file names, dataset distinguished by directory)
y_test_unet     = np.load(os.path.join(data_dir, 'y_test_masks_unet.npy')).flatten()
y_prob_unet     = np.load(os.path.join(data_dir, 'y_pred_probs_fire_unet.npy')).flatten()

# Sanity check: verify labels and probability ranges look correct
for name, y_true, probs in [
    ('Majority Class', y_test,      y_prob_majority),
    ('Random Forest',  y_test,      y_prob_rf),
    ('XGBoost',        y_test,      y_prob_xgb),
    ('U-Net',          y_test_unet, y_prob_unet),
]:
    print(f"{name}:")
    print(f"  Unique values y_true: {np.unique(y_true)}")
    print(f"  Probs — min: {probs.min():.3f}, max: {probs.max():.3f}, "
          f"unique count: {len(np.unique(probs))}")

# U-Net: low max probability is expected when trained with few epochs.
# The network has not learned to be confident in its predictions yet,
# causing the sigmoid output to stay low (well below 1.0).
# This will be resolved once the model is retrained on the HPC with more epochs and all data.

## Metric table at threshold 0.5 and 0.85

In [ ]:
def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'mIoU':      jaccard_score(y_true, y_pred, zero_division=0),
    }

models_table = {
    'Majority Class': (y_test,      y_prob_majority),
    'Random Forest':  (y_test,      y_prob_rf),
    'XGBoost':        (y_test,      y_prob_xgb),
    'U-Net':          (y_test_unet, y_prob_unet),
}

for t in [0.5, 0.85]:
    print(f"\n=== Metrics at Threshold {t} ===")
    print(f"{'Model':<20} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'mIoU':>8}")
    print("-" * 67)
    for name, (y_true, probs) in models_table.items():
        m = metrics_at_threshold(y_true, probs, t)
        print(f"{name:<20} {m['Accuracy']:>9.3f} {m['Precision']:>10.3f} "
              f"{m['Recall']:>8.3f} {m['F1']:>8.3f} {m['mIoU']:>8.3f}")

## mIoU vs Threshold

In [ ]:
models = {
    'Majority Class': (y_test,      y_prob_majority, 'tab:green',  'x', 2.5),
    'Random Forest':  (y_test,      y_prob_rf,       'tab:blue',   'o', 2.5),
    'XGBoost':         (y_test,     y_prob_xgb,       'tab:orange', 's', 2.5),
    'U-Net':           (y_test_unet, y_prob_unet,     'tab:red',    '^', 2.5),
}

plt.figure(figsize=(12, 7))

for name, (y_true, probs, color, marker, lw) in models.items():
    mious = [
        jaccard_score(y_true, (probs >= t).astype(int), zero_division=0)
        for t in THRESHOLDS
    ]
    plt.plot(THRESHOLDS, mious, label=name, linewidth=lw, marker=marker, color=color)

plt.axvline(0.5, color='gray', linestyle='--', label='Threshold 0.5 (standard)')
plt.title(f'{DATASET} Results: mIoU vs Threshold', fontsize=14, fontweight='bold')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('mIoU Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.ylim(0.0, 1.05)
plt.tight_layout()
plt.show()

## Precision & Recall vs Threshold

Shows the trade-off between precision and recall across all thresholds, per model.

In [ ]:
models_pr = {
    'Majority Class': (y_test,      y_prob_majority, 'tab:green'),
    'Random Forest':  (y_test,      y_prob_rf,       'tab:blue'),
    'XGBoost':        (y_test,      y_prob_xgb,      'tab:orange'),
    'U-Net':          (y_test_unet, y_prob_unet,     'tab:red'),
}

fig, axes = plt.subplots(1, len(models_pr), figsize=(20, 5), sharey=True)

for ax, (name, (y_true, probs, color)) in zip(axes, models_pr.items()):
    precisions = [precision_score(y_true, (probs >= t).astype(int), zero_division=0) for t in THRESHOLDS]
    recalls    = [recall_score(y_true,    (probs >= t).astype(int), zero_division=0) for t in THRESHOLDS]

    ax.plot(THRESHOLDS, precisions, color=color,                 label='Precision', linewidth=2)
    ax.plot(THRESHOLDS, recalls,    color=color, linestyle='--', label='Recall',    linewidth=2)
    ax.axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

axes[0].set_ylabel('Score', fontsize=12)
fig.suptitle(f'{DATASET}: Precision & Recall vs Threshold (per pixel)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Full classification reports at threshold 0.5 and 0.85

In [ ]:
for t in [0.5, 0.85]:
    print(f"\n=== {DATASET} Model Performance at Threshold {t} ===")
    for name, (y_true, probs) in models_table.items():
        y_pred = (probs >= t).astype(int)
        print(f"=== {name} ===")
        print(classification_report(
            y_true, y_pred,
            target_names=['Not burned (0)', 'Burned (1)'],
            zero_division=0
        ))
        print()